<a href="https://colab.research.google.com/github/luyov-belli/cv_experience_ocr_bert/blob/chore%2Focr-default-v6/notebooks/colab_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# cv_experience_ocr_bert: etapas pesadas en Colab

Este notebook **no contiene lógica propia**: clona el repo y ejecuta los scripts de `src/`. Todo cambio de código se hace en la laptop, va a una rama y entra a `main` por PR. Aquí solo se hace `git pull`.

Las salidas pesadas (imágenes, OCR, pesos) se guardan en Google Drive mediante enlaces simbólicos, porque la máquina de Colab se borra al cerrar la sesión.

**Antes de empezar:**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4* (la descarga no la necesita, pero el OCR y BERT sí).
2. Panel izquierdo → 🔑 *Secrets* → crear `HF_TOKEN` con tu token de Hugging Face y activar *Notebook access*.

## 0. Preparación (correr siempre al abrir el notebook)

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'Sin GPU'

from google.colab import drive
drive.mount('/content/drive')

name, memory.total [MiB]
Tesla T4, 15360 MiB
Mounted at /content/drive


In [2]:
# Token de Hugging Face: los Secrets solo se leen desde el kernel del notebook,
# no desde los scripts lanzados con `!python ...` (otro proceso). Se copia a una
# variable de entorno, que esos procesos heredan.
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('HF_TOKEN cargado:', bool(os.environ['HF_TOKEN']))

HF_TOKEN cargado: True


In [3]:
import os

REPO_URL = 'https://github.com/luyov-belli/cv_experience_ocr_bert'
REPO = '/content/cv_experience_ocr_bert'
DRIVE = '/content/drive/MyDrive/cv_experience_ocr_bert'
BRANCH = 'main'  # para probar una rama antes de fusionarla, cambia esto

if not os.path.exists(REPO):
    !git clone -q {REPO_URL} {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!git log --oneline -1

# Carpetas pesadas -> Drive (persisten entre sesiones)
for sub in ['data/raw', 'data/ocr', 'models/bert']:
    target = f'{DRIVE}/{sub}'
    os.makedirs(target, exist_ok=True)
    if not os.path.islink(sub):
        !rm -rf {sub}
        os.symlink(target, sub)
!ls -la data models

/content/cv_experience_ocr_bert
bf5da9c (HEAD -> main, origin/main, origin/HEAD) Merge pull request #7 from luyov-belli/feature/ocr-extraction
data:
total 444
drwxr-xr-x 4 root root   4096 Oct 26 12:21 .
drwxr-xr-x 9 root root   4096 Oct 26 12:21 ..
lrwxrwxrwx 1 root root     54 Oct 26 12:21 ocr -> /content/drive/MyDrive/cv_experience_ocr_bert/data/ocr
drwxr-xr-x 2 root root   4096 Oct 26 12:21 processed
lrwxrwxrwx 1 root root     54 Oct 26 12:21 raw -> /content/drive/MyDrive/cv_experience_ocr_bert/data/raw
-rw-r--r-- 1 root root 437136 Oct 26 12:21 sampled_ids.csv
drwxr-xr-x 2 root root   4096 Oct 26 12:21 splits

models:
total 12
drwxr-xr-x 3 root root 4096 Oct 26 12:21 .
drwxr-xr-x 9 root root 4096 Oct 26 12:21 ..
drwxr-xr-x 2 root root 4096 Oct 26 12:21 baselines
lrwxrwxrwx 1 root root   57 Oct 26 12:21 bert -> /content/drive/MyDrive/cv_experience_ocr_bert/models/bert


In [4]:
# Dependencias (ver requirements-colab.txt). Se congelan en la versión que trae
# Colab solo las librerías de las que dependen sus herramientas internas:
#   pandas -> google-colab exige su versión exacta
#   numpy  -> numba y buena parte del stack de Colab
#   torch  -> build con CUDA de Colab
# pyarrow NO se congela: datasets 5.1 exige pyarrow>=24 y lo único en Colab que
# pide <24 es cudf, que este proyecto no usa.
# Si algo exigiera otra versión de las congeladas, pip falla aquí de forma
# explícita en vez de romper Colab en silencio.
import numpy, pandas, torch
keep = [f'{m.__name__}=={m.__version__.split("+")[0]}' for m in (numpy, pandas, torch)]
with open('/tmp/colab_keep.txt', 'w') as f:
    f.write('\n'.join(keep))
print('Se conservan:', keep)
!pip install -q -r requirements-colab.txt -c /tmp/colab_keep.txt

Se conservan: ['numpy==2.1.3', 'pandas==2.2.3', 'torch==2.11.0']
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 2.3.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


## 1. Descarga de datos (una sola vez)

Muestrea 800 CVs sin foto por ocupación (semilla 42) y guarda las imágenes en Drive. Toma del orden de 15–30 minutos. Si se corta, vuelve a correrla: el muestreo es determinista y produce la misma muestra.

In [5]:
!python -m src.download_data

README.md: 100% 6.65k/6.65k [00:00<00:00, 13.6MB/s]
accountants_auditors: 100% 800/800 [04:19<00:00,  3.09it/s]
construction_laborers: 100% 800/800 [04:01<00:00,  3.31it/s]
elementary_middle_school_teachers: 100% 800/800 [04:26<00:00,  3.00it/s]
registered_nurses: 100% 800/800 [04:20<00:00,  3.07it/s]
software_developers: 100% 800/800 [04:16<00:00,  3.12it/s]

data/text/accountants_auditors.parquet: downloading bytes:   5% 78.1k/1.71M [00:00<00:17, 94.6kB/s]
data/text/accountants_auditors.parquet: downloading bytes:  13% 228k/1.71M [00:00<00:04, 302kB/s, 7.65kB/s  ]
data/text/accountants_auditors.parquet: downloading bytes:  63% 1.08M/1.71M [00:01<00:00, 1.64MB/s, 47.1kB/s  ]
data/text/accountants_auditors.parquet: downloading bytes: 100% 1.70M/1.70M [00:01<00:00, 1.18MB/s,  162kB/s  ]
data/text/accountants_auditors.parquet: reconstructing file: 100% 1.71M/1.71M [00:01<00:00, 1.19MB/s,  163kB/s  ]

data/text/construction_laborers.parquet: downloading bytes:   0% 0.00/1.16M [00:00<?, ?B

In [6]:
# sampled_ids.csv vive en el repo (se versiona). Se copia a Drive para que
# lo bajes a la laptop y lo subas en un commit.
!cp data/sampled_ids.csv {DRIVE}/data/sampled_ids.csv
!ls {DRIVE}/data/raw | head; ls {DRIVE}/data/raw/images | wc -l

images
metadata.csv
reference_text.parquet
4000


## 2. OCR con PaddleOCR (GPU)

**Correr en una sesión distinta a la de BERT.** PaddlePaddle-GPU y PyTorch traen cada uno sus propias librerías de NVIDIA, y al instalar una puede cambiar la versión que usa la otra. Para el OCR, abre una sesión nueva (*Entorno de ejecución → Desconectar y borrar*), corre la sección 0 y luego esta.

In [7]:
# PaddlePaddle-GPU del mismo número de versión que requirements.txt (3.3.1), en el
# build que corresponde al CUDA que trae Colab.
import torch
major, minor = map(int, torch.version.cuda.split('.')[:2])
tag = 'cu130' if major >= 13 else 'cu129' if (major, minor) >= (12, 8) else 'cu126' if major == 12 else 'cu118'
print('CUDA de Colab:', torch.version.cuda, '-> índice', tag)
!pip install -q paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/{tag}/
!python -c "import paddle; print('paddle', paddle.__version__, '| GPUs:', paddle.device.cuda.device_count())"

CUDA de Colab: 13.0 -> índice cu130
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0+cu130 requires nvidia-cudnn-cu13==9.19.0.56; platform_system == "Linux", but you have nvidia-cudnn-cu13 9.13.0.50 which is incompatible.
torch 2.11.0+cu130 requires nvidia-cusparselt-cu13==0.8.0; platform_system == "Linux", but you have nvidia-cusparselt-cu13 0.8.1 which is incompatible.
torch 2.11.0+cu130 requires nvidia-nccl-cu13==2.28.9; platform_system == "Linux", but you have nvidia-nccl-cu13 2.28.3 which is incompatible.
/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
paddle 3.3.1 | GPUs: 1

### 2a. Elegir el modelo (decisión C)

Corre los 3 modelos (PP-OCRv5 server, PP-OCRv5 mobile, PP-OCRv6 medium) sobre los mismos 50 CVs (10 por ocupación) y mide tiempo, CER, WER y confianza contra el texto de referencia. No toca la caché de `data/ocr/`.

In [8]:
!python -m src.ocr_extraction --compare 10
!mkdir -p {DRIVE}/output/metrics && cp output/metrics/ocr_model_comparison*.csv {DRIVE}/output/metrics/

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/content/cv_experience_ocr_bert/src/ocr_extraction.py:69: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  return PaddleOCR(lang="en", text_detection_model_name=det,
Creating model: ('PP-OCRv5_server_det', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv5_server_det`.
[2026-10-26 12:51:34,621] [    INFO] _client.py:1085 - HTTP Request: GET https://huggingface.c

### 2b. OCR de los 4,000 CVs

Pon en `OCR_MODEL` el modelo elegido en 2a (`server`, `mobile` o `v6`). Guarda un JSON por CV en Drive; si la sesión se corta, vuelve a correr la celda y retoma donde quedó.

In [9]:
OCR_MODEL = 'v6'  # cambiar según el resultado de 2a
!python -m src.ocr_extraction --model {OCR_MODEL}

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/content/cv_experience_ocr_bert/src/ocr_extraction.py:69: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  return PaddleOCR(lang="en", text_detection_model_name=det,
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.
OCR v6: 100% 4000/4000 [49:57<00:00,  

In [10]:
# Métricas del OCR a Drive, y los 4,000 JSON en un solo zip para bajarlos a la laptop.
!cp output/metrics/ocr_stats.csv output/metrics/ocr_failures.csv output/metrics/ocr_run_summary.json {DRIVE}/output/metrics/
!cd data && zip -qr {DRIVE}/ocr_json.zip ocr && ls -lh {DRIVE}/ocr_json.zip
!ls {DRIVE}/data/ocr | wc -l

-rw------- 1 root root 4.4M Oct 26 13:48 /content/drive/MyDrive/cv_experience_ocr_bert/ocr_json.zip
4000
